# Fraud Detection Model v3 (FINAL ACTUAL - DO NOT TOUCH)
**Author:** Dave (Lead DS)  
**Date:** Nov 2026  
**Notes:** Tuned for Black Friday. Getting 0.994 AUC on validation!! Ship `fraud_v3_final.pkl` to backend team ASAP.

In [ ]:
# CRIME #1: Hardcoded local paths, unpinned imports, and out-of-order execution
from pathlib import Path
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pickle

# Hardcoded path on Dave's local machine (checks repo ../data/raw/ path or synthetic fallback in lab)
LOCAL_DUMP_PATH = Path("/Users/dave_ds/Desktop/dump_oct_final_v2.csv")
REPO_DATA_PATH = Path("../data/raw/dump_oct_final_v2.csv")

if LOCAL_DUMP_PATH.exists():
    df = pd.read_csv(LOCAL_DUMP_PATH)
elif REPO_DATA_PATH.exists():
    df = pd.read_csv(REPO_DATA_PATH)
    print(f"Loaded {len(df)} rows from dump_oct_final_v2.csv!")
else:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        "transaction_id": [f"tx_{i:06d}" for i in range(1, n + 1)],
        "user_id": np.random.choice([f"u_{i:04d}" for i in range(1, 121)], size=n),
        "timestamp": pd.date_range("2026-10-01 08:00:00", periods=n, freq="2min"),
        "amount": np.random.exponential(scale=65.0, size=n).round(2),
        "transaction_currency": np.random.choice(["USD", "EUR", "GBP", "840"], size=n, p=[0.831, 0.097, 0.034, 0.038]),
        "merchant_risk_score": np.where(np.random.rand(n) < 0.058, np.nan, np.random.beta(2, 5, size=n)),
        "is_fraud": np.random.choice([0, 1], size=n, p=[0.95, 0.05])
    })
    print(f"Loaded {len(df)} rows from synthetic fallback!")

Loaded 1000 rows from dump_oct_final_v2.csv!


In [ ]:
# CRIME #2: Global state mutation & silent currency mapping without schema validation
currency_map = {"USD": 1.0, "EUR": 1.08, "GBP": 1.27}

# If upstream sends ISO numeric code (e.g. '840') instead of 'USD', .map() produces NaN -> filled with 0.0!
df["currency_rate"] = df["transaction_currency"].astype(str).map(currency_map).fillna(0.0)
df["amount_usd"] = df["amount"] * df["currency_rate"]

In [ ]:
# CRIME #3: Severe Data Leakage (Future Peeking + Pre-Split Global Target/Feature Leakage)
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

# BUG A: center=True looks into FUTURE transactions to compute current velocity!
df["tx_count_5m"] = df.groupby("user_id")["amount_usd"].transform(
    lambda x: x.rolling(5, center=True, min_periods=1).count()
)

# BUG B: Leaking the ground-truth label (`is_fraud`) directly into a feature via global target mean!
df["user_historical_fraud_rate"] = df.groupby("user_id")["is_fraud"].transform("mean")

# BUG C: Imputing missing values across the ENTIRE dataset before train/test split
df["merchant_risk_score"] = df["merchant_risk_score"].fillna(df["merchant_risk_score"].mean())
print("Velocity features & global imputation complete.")

Velocity features & global imputation complete.


In [ ]:
# CRIME #4: Random train_test_split (shuffle=True) on time-series data + wrong metric (ROC-AUC on imbalanced data)
features = ["amount_usd", "currency_rate", "tx_count_5m", "merchant_risk_score", "user_historical_fraud_rate"]
X = df[features]
y = df["is_fraud"]

# Shuffling time-series data mixes future transactions into historical training data!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

# Magic numbers hardcoded directly in notebook cell
model = xgb.XGBClassifier(n_estimators=100, max_depth=9, learning_rate=0.1)
model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:, 1]
print(f"Validation ROC-AUC: {roc_auc_score(y_test, preds):.4f}")

Validation ROC-AUC: 0.9942


In [ ]:
# CRIME #5: Insecure Pickle serialization & zero model signature / input contract
pickle.dump(model, open("fraud_v3_final.pkl", "wb"))
print("Saved fraud_v3_final.pkl! Ready to email to backend team.")

Saved fraud_v3_final.pkl! Ready to email to backend team.
